In [2]:
import requests
import pandas as pd
from datetime import datetime
from typing import List, Dict
import os

def fetch_weekly_data(base_url: str, prefecture: str, start_date: str, end_date: str) -> List[Dict]:
    """
    Holt wöchentliche Kraftstoffpreise für eine Präfektur und Zeiträume.
    
    Args:
        base_url: API-Basis-URL (z.B. 'http://127.0.0.1:8000/api/data/weekly/prefecture')
        prefecture: Präfektur-Name (z.B. 'ATTICA')
        start_date: Startdatum (YYYY-MM-DD)
        end_date: Enddatum (YYYY-MM-DD)
    
    Returns:
        Liste von Datensätzen
    """
    url = f"{base_url}/{prefecture}"
    params = {
        'start_date': start_date,
        'end_date': end_date,
        'accept': 'application/json'
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    return response.json()

def flatten_fuel_data(data_list: List[Dict]) -> pd.DataFrame:
    """
    Flacht die verschachtelte JSON-Struktur zu einem DataFrame ab.
    
    Args:
        data_list: Rohdaten von der API
    
    Returns:
        Pandas DataFrame mit Spalten: date, fuel_type, price, data_file
    """
    rows = []
    
    for entry in data_list:
        date = entry['date']
        data_file = entry.get('data_file', '')
        
        for fuel_entry in entry['data']:
            row = {
                'date': date,
                'fuel_type': fuel_entry['fuel_type'],
                'price': float(fuel_entry['price']),
                'data_file': data_file
            }
            rows.append(row)
    
    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values(['fuel_type', 'date']).reset_index(drop=True)

def save_dataframe(df: pd.DataFrame, filename: str = None):
    """
    Speichert DataFrame als Parquet (effizient) und CSV (lesbar).
    """
    if filename is None:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"data/fuel_prices_{timestamp}"
    
    #parquet_file = f"{filename}.parquet"
    csv_file = f"{filename}.csv"
    
    # Parquet für effiziente Speicherung/Laden
    #df.to_parquet(parquet_file, index=False)
    #print(f"✅ Parquet gespeichert: {parquet_file} ({df.shape[0]:,} Zeilen, {df.shape[1]} Spalten)")
    
    # CSV für einfaches Öffnen
    df.to_csv(csv_file, index=False)
    print(f"✅ CSV gespeichert: {csv_file}")
    
    # Zusätzliche Infos
    print(f"\n📊 Übersicht:")
    print(df.groupby('fuel_type').size())
    print(f"\nZeitraum: {df['date'].min()} bis {df['date'].max()}")
    print(f"Präfektur: {df['data_file'].iloc[0] if 'data_file' in df.columns else 'N/A'}")



In [ ]:
import requests
import pandas as pd
from datetime import datetime
from typing import List, Dict
import json
import os
from tqdm import tqdm  # Für Fortschrittsbalken

def load_prefectures(prefecture_file: str) -> List[str]:
    """
    Lädt Präfektur-Namen aus JSON-Datei.
    
    Args:
        prefecture_file: Pfad zur JSON-Datei
    
    Returns:
        Liste der Präfektur-Namen (name-Feld)
    """
    with open(prefecture_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    prefectures = [prefecture['name'] for prefecture in data]
    print(f"📍 {len(prefectures)} Präfekturen geladen: {prefectures[:5]}...")
    return prefectures

def fetch_weekly_data(base_url: str, prefecture: str, start_date: str, end_date: str) -> List[Dict]:
    """Holt wöchentliche Daten für eine Präfektur (wie vorher)."""
    url = f"{base_url}/{prefecture}"
    params = {'start_date': start_date, 'end_date': end_date}
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as e:
        print(f"❌ Fehler bei {prefecture}: {e}")
        return []

def flatten_and_filter_fuel_data(data_list: List[Dict], prefecture: str, fuel_types: List[str] = ['DIESEL', 'SUPER', "Diesel", "Super"]) -> pd.DataFrame:
    """
    Flacht JSON und filtert nur DIESEL + SUPER.
    """
    rows = []
    
    for entry in data_list:
        date = entry['date']
        #data_file = entry.get('data_file', '')
        
        for fuel_entry in entry['data']:
            fuel_type = fuel_entry['fuel_type']
            if fuel_type in fuel_types:
                rows.append({
                    'prefecture': prefecture,  # Neu: Präfektur als Spalte
                    'date': date,
                    'fuel_type': fuel_type,
                    'price': float(fuel_entry['price']),
                    #'data_file': data_file
                })
    
    if rows:
        df = pd.DataFrame(rows)
        df['date'] = pd.to_datetime(df['date'])
        return df.sort_values(['prefecture', 'fuel_type', 'date']).reset_index(drop=True)
    return pd.DataFrame()

def fetch_all_prefectures(base_url: str, prefectures: List[str], start_date: str, end_date: str) -> pd.DataFrame:
    """Holt Daten für ALLE Präfekturen."""
    all_data = []
    
    for prefecture in tqdm(prefectures, desc="Präfekturen laden"):
        raw_data = fetch_weekly_data(base_url, prefecture, start_date, end_date)
        df_pref = flatten_and_filter_fuel_data(raw_data, prefecture=prefecture)
        if not df_pref.empty:
            all_data.append(df_pref)
    
    if all_data:
        return pd.concat(all_data, ignore_index=True)
    return pd.DataFrame()

def save_dataframe(df: pd.DataFrame, filename: str = None):
    """Speichert als Parquet + CSV (wie vorher)."""
    if filename is None:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"fuel_prices_all_diesel_super_{timestamp}"
    
    #parquet_file = f"{filename}.parquet"
    csv_file = f"{filename}.csv"
    
    #df.to_parquet(parquet_file, index=False)
    df.to_csv(csv_file, index=False)
    
    #print(f"\n✅ Parquet: {parquet_file} ({df.shape[0]:,} Zeilen)")
    print(f"✅ CSV: {csv_file}")
    print(f"\n📊 {len(df['prefecture'].unique())} Präfekturen, {df['date'].min()} - {df['date'].max()}")

# ═══════════════════════════════════════════════════════════════


In [ ]:
# KONFIGURATION - HIER ANPASSEN
# ═══════════════════════════════════════════════════════════════
PREFECTURE_JSON = "data/prefecture.json"  # Deine JSON-Datei
BASE_URL = "http://127.0.0.1:8000/api/data/weekly/prefecture"
START_DATE = "2010-01-01"
END_DATE = "2020-12-31"  # Länger für mehr Daten
FUEL_TYPES = ["DIESEL", "SUPER", "Diesel", "Super"]  # Nur diese behalten


print("🚀 Lade Präfekturen...")
prefectures = load_prefectures(PREFECTURE_JSON)
    
print("📡 Hole Daten für alle Präfekturen (DIESEL + SUPER)...")
df_all = fetch_all_prefectures(BASE_URL, prefectures, START_DATE, END_DATE)
    
if not df_all.empty:
    print(f"\n📊 Gesamt: {df_all.shape[0]:,} Preise aus {len(df_all['prefecture'].unique())} Präfekturen")
    save_dataframe(df_all)
        
    print("\n🔍 Erste Zeilen:")
    print(df_all.head(10))
    print("\n👀 Nach Präfektur:")
    print(df_all.groupby(['prefecture', 'fuel_type']).size().head(10))
else:
    print("❌ Keine Daten gefunden!")
    
print("\n🎉 Fertig!")

🚀 Lade Präfekturen...
📍 51 Präfekturen geladen: ['ATTICA', 'AETOLIA_ACARNANIA', 'ARGOLIS', 'ARKADIAS', 'ARTA']...
📡 Hole Daten für alle Präfekturen (DIESEL + SUPER)...


Präfekturen laden: 100%|██████████| 51/51 [00:00<00:00, 122.58it/s]


📊 Gesamt: 2,448 Preise aus 51 Präfekturen
✅ CSV: fuel_prices_all_diesel_super_20260416_213623.csv

📊 51 Präfekturen, 2023-01-06 00:00:00 - 2023-12-29 00:00:00

🔍 Erste Zeilen:
  prefecture       date fuel_type  price
0     ATTICA 2023-01-06    DIESEL  1.768
1     ATTICA 2023-01-13    DIESEL  1.768
2     ATTICA 2023-01-20    DIESEL  1.767
3     ATTICA 2023-01-27    DIESEL  1.795
4     ATTICA 2023-02-03    DIESEL  1.758
5     ATTICA 2023-02-10    DIESEL  1.686
6     ATTICA 2023-02-17    DIESEL  1.686
7     ATTICA 2023-03-03    DIESEL  1.673
8     ATTICA 2023-03-10    DIESEL  1.686
9     ATTICA 2023-03-17    DIESEL  1.663

👀 Nach Präfektur:
prefecture         fuel_type
ACHAEA             DIESEL       48
AETOLIA_ACARNANIA  DIESEL       48
ARGOLIS            DIESEL       48
ARKADIAS           DIESEL       48
ARTA               DIESEL       48
ATTICA             DIESEL       48
BOEOTIA            DIESEL       48
CEPHALONIA         DIESEL       48
CHALKIDIKI         DIESEL       48
CHANIA   

In [21]:
# ═══════════════════════════════════════════════════════════════
# KONFIGURATION - HIER ANPASSEN
# ═══════════════════════════════════════════════════════════════
BASE_URL = "http://127.0.0.1:8000/api/data/weekly/prefecture"
PREFECTURE = "ATTICA"
START_DATE = "2010-01-01"
END_DATE = "2019-12-31"


print("🚀 Hole Daten von der FuelPrices API...")
    
    # 1. Daten abrufen
raw_data = fetch_weekly_data(BASE_URL, PREFECTURE, START_DATE, END_DATE)
print(f"📥 {len(raw_data)} Datensätze abgerufen")
    
    # 2. In DataFrame umwandeln
df = flatten_fuel_data(raw_data)
print(f"📊 DataFrame erstellt: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
    
    # 3. Speichern
save_dataframe(df, "greece_data_all_2019")
    
    # 4. Erste Zeilen anzeigen
print("\n🔍 Erste 10 Zeilen:")
print(df.head(10))
    
print("\n🎉 Fertig!")

🚀 Hole Daten von der FuelPrices API...
📥 52 Datensätze abgerufen
📊 DataFrame erstellt: 284 Zeilen, 4 Spalten
✅ CSV gespeichert: greece_data_all_2019.csv

📊 Übersicht:
fuel_type
DIESEL            52
DIESEL_HEATING    24
GAS               52
SUPER             52
UNLEADED_100      52
UNLEADED_95       52
dtype: int64

Zeitraum: 2019-01-04 00:00:00 bis 2019-12-27 00:00:00
Präfektur: http://www.fuelprices.gr/files/deltia/EBDOM_DELTIO_04_01_2019.pdf

🔍 Erste 10 Zeilen:
        date fuel_type  price  \
0 2019-01-04    DIESEL  1.297   
1 2019-01-11    DIESEL  1.298   
2 2019-01-18    DIESEL  1.306   
3 2019-01-25    DIESEL  1.311   
4 2019-02-01    DIESEL  1.312   
5 2019-02-08    DIESEL  1.316   
6 2019-02-15    DIESEL  1.326   
7 2019-02-22    DIESEL  1.349   
8 2019-03-01    DIESEL  1.357   
9 2019-03-08    DIESEL  1.361   

                                           data_file  
0  http://www.fuelprices.gr/files/deltia/EBDOM_DE...  
1  http://www.fuelprices.gr/files/deltia/EBDOM_DE...  
2  

In [4]:
os.makedirs(os.path.join("data", "prefecture_data"), exist_ok=True)
BASE_URL = "http://127.0.0.1:8000/api/data/weekly/prefecture"
START_DATE = "2010-01-01"

PREFECTURES_DF = pd.read_json(os.path.join("data","prefecture.json"))
PREFECTURES = PREFECTURES_DF.loc[:, "name"]

YEARS = [str(2010+k) for k in range(11)]

print("🚀 Hole Daten von der FuelPrices API...")
for prefecture in PREFECTURES:
    data_length = 0
    pref_df = pd.DataFrame()
    for year in YEARS:
        END_DATE = f"{year}-12-31"
        raw_data = fetch_weekly_data(BASE_URL, prefecture, START_DATE, END_DATE)
        if len(raw_data) == 0:
            continue
        data_length = data_length + len(raw_data)
        df = flatten_fuel_data(raw_data)
        pref_df = pd.concat([pref_df, df])
    
    print(f"📥 {data_length} Datensätze für {prefecture} abgerufen")
    print(f"📊 DataFrame erstellt: {pref_df.shape[0]} Zeilen, {pref_df.shape[1]} Spalten")
    save_dataframe(pref_df, os.path.join("data", "prefecture_data", f"data_{prefecture}"))
print("\n🎉 Fertig!")

🚀 Hole Daten von der FuelPrices API...
📥 445 Datensätze für ATTICA abgerufen
📊 DataFrame erstellt: 2425 Zeilen, 4 Spalten
✅ CSV gespeichert: data/prefecture_data/data_ATTICA.csv

📊 Übersicht:
fuel_type
DIESEL            445
DIESEL_HEATING    200
GAS               445
SUPER             445
UNLEADED_100      445
UNLEADED_95       445
dtype: int64

Zeitraum: 2012-05-04 00:00:00 bis 2020-12-25 00:00:00
Präfektur: http://www.fuelprices.gr/files/deltia/EBDOM_DELTIO_04_05_2012.pdf
📥 445 Datensätze für AETOLIA_ACARNANIA abgerufen
📊 DataFrame erstellt: 2347 Zeilen, 4 Spalten
✅ CSV gespeichert: data/prefecture_data/data_AETOLIA_ACARNANIA.csv

📊 Übersicht:
fuel_type
DIESEL            445
DIESEL_HEATING    200
GAS               422
SUPER             390
UNLEADED_100      445
UNLEADED_95       445
dtype: int64

Zeitraum: 2012-05-04 00:00:00 bis 2020-12-25 00:00:00
Präfektur: http://www.fuelprices.gr/files/deltia/EBDOM_DELTIO_04_05_2012.pdf
📥 433 Datensätze für ARGOLIS abgerufen
📊 DataFrame erstellt

In [3]:
import os
len(os.listdir(os.path.join("data", "prefecture_data")))
os.listdir(os.path.join("data", "prefecture_data"))

['data_EVRYTANIA.csv',
 'data_BOEOTIA.csv',
 'data_KILKIS.csv',
 'data_KARDITSA.csv',
 'data_DODECANESE.csv',
 'data_ARKADIAS.csv',
 'data_SAMOS.csv',
 'data_MAGNESIA.csv',
 'data_CHIOS.csv',
 'data_RHODOPE.csv',
 'data_AETOLIA_ACARNANIA.csv',
 'data_PIERIA.csv',
 'data_THESSALONIKI.csv',
 'data_LEFKADA.csv',
 'data_RETHYMNO.csv',
 'data_ARTA.csv',
 'data_GREVENA.csv',
 'data_ATTICA.csv',
 'data_CHANIA.csv',
 'data_KAVALA.csv',
 'data_ARGOLIS.csv',
 'data_KASTORIA.csv',
 'data_THESPROTIA.csv',
 'data_LESBOS.csv',
 'data_CHALKIDIKI.csv',
 'data_TRIKALA.csv',
 'data_IMATHIA.csv',
 'data_XANTHI.csv',
 'data_MESSENIA.csv',
 'data_PELLA.csv',
 'data_ZAKYNTHOS.csv',
 'data_DRAMA.csv',
 'data_PHOCIS.csv',
 'data_CEPHALONIA.csv',
 'data_LARISSA.csv',
 'data_HERAKLION.csv',
 'data_CYCLADES.csv',
 'data_ACHAEA.csv',
 'data_PHTHIOTIS.csv',
 'data_CORINTHIA.csv',
 'data_KERKYRA.csv',
 'data_EVROS.csv',
 'data_KOZANI.csv',
 'data_LASITHI.csv',
 'data_FLORINA.csv',
 'data_LACONIA.csv',
 'data_SERRES